# Prepping Data Challenge - Week 20
**SuperBytes** customer Data

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [2]:
customer_spending_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_20_Customer_Data.xlsx', sheet_name='Customer Spending')
customer_name_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_20_Customer_Data.xlsx', sheet_name='Customer Names')

In [3]:
customer_spending_df.head()

,First,Last,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,Name,Name,Gender,Receipt Number,Date,Online,In Person,Sale Total
1,Emeline,Woollard,Female,2578226121,2023-12-27 00:00:00,No,Yes,250.54
2,Wallace,Slyde,Male,6574211891,2023-08-06 00:00:00,No,Yes,363.43
3,Chrissy,MacGiany,Male,7742449501,2023-04-19 00:00:00,No,Yes,487.25
4,Jacques,Brauns,Male,3112564839,2023-12-25 00:00:00,No,Yes,195.67


### Fix customer_spending column name

In [4]:
# Add correct column names
customer_spending_df.columns = ['First Name', 'Last Name', 'Gender', 'Receipt Number', 'Date', 'Online', 'In Person', 'Sale Total']

In [5]:
# Remove the first row (use loc instead of iloc to ensure idempotency)
customer_spending_df = customer_spending_df.loc[1:, :]

### Continue exploring Data

In [6]:
customer_spending_df.head()

,First Name,Last Name,Gender,Receipt Number,Date,Online,In Person,Sale Total
1,Emeline,Woollard,Female,2578226121,2023-12-27 00:00:00,No,Yes,250.54
2,Wallace,Slyde,Male,6574211891,2023-08-06 00:00:00,No,Yes,363.43
3,Chrissy,MacGiany,Male,7742449501,2023-04-19 00:00:00,No,Yes,487.25
4,Jacques,Brauns,Male,3112564839,2023-12-25 00:00:00,No,Yes,195.67
5,Joannes,Cabrera,Female,8018081174,2023-07-20 00:00:00,No,Yes,95.1


In [7]:
customer_name_df.head()

,First Name,Last Name,Customer ID
0,Emeline,Woollard,19-562-5181
1,Wallace,Slyde,64-198-3761
2,Chrissy,MacGiany,99-390-6705
3,Jacques,Brauns,19-284-3677
4,Joannes,Cabrera,46-687-7158


In [8]:
customer_name_df['Customer ID'].nunique()

999

In [9]:
customer_name_df.shape

(999, 3)

## Challenges

### Join tables

In [10]:
customer_df = customer_name_df.merge(customer_spending_df, on=['First Name', 'Last Name'])

In [11]:
customer_df.head()

,First Name,Last Name,Customer ID,Gender,Receipt Number,Date,Online,In Person,Sale Total
0,Emeline,Woollard,19-562-5181,Female,2578226121,2023-12-27 00:00:00,No,Yes,250.54
1,Wallace,Slyde,64-198-3761,Male,6574211891,2023-08-06 00:00:00,No,Yes,363.43
2,Chrissy,MacGiany,99-390-6705,Male,7742449501,2023-04-19 00:00:00,No,Yes,487.25
3,Jacques,Brauns,19-284-3677,Male,3112564839,2023-12-25 00:00:00,No,Yes,195.67
4,Joannes,Cabrera,46-687-7158,Female,8018081174,2023-07-20 00:00:00,No,Yes,95.1


### Remove Names for privacy

In [12]:
customer_df = customer_df.drop(columns=['First Name', 'Last Name'])

### Combine Online and In Person columns into one

We can do this cleanly, with `idxmax()`

In [13]:
# Turn columns to pseudo boolean for comparison 
customer_df['Online'] = customer_df['Online'].map({'No' : 0, 'Yes' : 1})
customer_df['In Person'] = customer_df['In Person'].map({'No' : 0, 'Yes' : 1})

In [14]:
# Build categorical column
customer_df['Online or In Person'] = customer_df[['Online', 'In Person']].idxmax(axis=1)

In [15]:
# Drop 'Online' and 'In Person' columns
customer_df = customer_df.drop(columns=['Online', 'In Person'])

### Add a Field for Day of the Week

In [16]:
customer_df.head()

,Customer ID,Gender,Receipt Number,Date,Sale Total,Online or In Person
0,19-562-5181,Female,2578226121,2023-12-27 00:00:00,250.54,In Person
1,64-198-3761,Male,6574211891,2023-08-06 00:00:00,363.43,In Person
2,99-390-6705,Male,7742449501,2023-04-19 00:00:00,487.25,In Person
3,19-284-3677,Male,3112564839,2023-12-25 00:00:00,195.67,In Person
4,46-687-7158,Female,8018081174,2023-07-20 00:00:00,95.1,In Person


In [17]:
# Cast Date Column as DateTime
customer_df['Date'] = pd.to_datetime(customer_df['Date'])

In [18]:
# Extract Weekday
customer_df['Weekday'] = customer_df['Date'].dt.day_name()

In [19]:
# Remove Date Column
customer_df = customer_df.drop(columns='Date')

### Separate Data Into different DataFrames based on weekday

In [20]:
sales_by_weekday = {weekday : customer_df[customer_df['Weekday'] == weekday] for weekday in customer_df['Weekday'].unique()}

### Rank sales for each weekday

In [21]:
for weekday in sales_by_weekday:
    
    # Cast sales as float
    sales_by_weekday[weekday].loc[: , 'Sale Total'] = sales_by_weekday[weekday].loc[: , 'Sale Total'].astype(float)

    # Sort by sales total
    sales_by_weekday[weekday] = sales_by_weekday[weekday].sort_values(by='Sale Total', ascending=False)

    # Add a ranking column
    sales_by_weekday[weekday]['Rank'] = range(1, sales_by_weekday[weekday].shape[0] + 1)

## Output

In [22]:
sales_by_weekday['Monday'].head()

,Customer ID,Gender,Receipt Number,Sale Total,Online or In Person,Weekday,Rank
850,39-459-7177,Male,8382576885,494.6,Online,Monday,1
48,58-100-9851,Female,9293958996,493.63,Online,Monday,2
296,55-058-7747,Female,2498898174,492.2,Online,Monday,3
158,56-570-2703,Male,3765914924,491.96,In Person,Monday,4
119,37-000-7501,Female,3806822042,490.4,In Person,Monday,5


In [23]:
sales_by_weekday['Tuesday'].head()

,Customer ID,Gender,Receipt Number,Sale Total,Online or In Person,Weekday,Rank
635,64-659-1058,Female,6835837217,498.13,Online,Tuesday,1
334,01-898-3652,Female,4535345082,494.47,Online,Tuesday,2
68,78-948-2572,Female,6353744226,494.08,In Person,Tuesday,3
148,17-510-6267,Male,670823538,492.19,Online,Tuesday,4
319,88-440-2735,Male,3500300413,490.73,Online,Tuesday,5


In [24]:
sales_by_weekday['Wednesday'].head()

,Customer ID,Gender,Receipt Number,Sale Total,Online or In Person,Weekday,Rank
125,83-513-5479,Male,6359624222,497.73,In Person,Wednesday,1
480,83-320-3342,Female,1443310328,488.13,In Person,Wednesday,2
2,99-390-6705,Male,7742449501,487.25,In Person,Wednesday,3
81,51-726-9424,Male,6878721954,484.92,In Person,Wednesday,4
579,12-054-0274,Female,898996864,481.35,In Person,Wednesday,5


## Save Output

In [25]:
for weekday in sales_by_weekday:
    sales_by_weekday[weekday].to_csv(f'D:01_Projekt_Portfolio/data_prepping_challenges/outputs/20_sales_{weekday}.csv')